In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm
import pickle
import matplotlib.pyplot as plt
import random

# --- 1. 설정 (Configuration) ---
DATASET_MODE = "CHEXPERT"  # "NIH" 또는 "CHEXPERT" 선택
BASE_PATH = '/kaggle/input/datasets/yoodeoksu/x-ray-s320/' # 실제 데이터 경로

# 공통 질병 5종 (성능 비교용)
COMMON_5 = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Pneumonia', 'Pneumothorax']

CONFIG = {
    "NIH": {
        "train_dir": os.path.join(BASE_PATH, "nih_train_320"),
        "val_dir": os.path.join(BASE_PATH, "nih_val_320"),
        "train_csv": os.path.join(BASE_PATH, "nih_train_320.csv"),
        "val_csv": os.path.join(BASE_PATH, "nih_val_320.csv"),
        "labels": ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 
                   'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 
                   'Fibrosis', 'Pleural_Thickening', 'Hernia'],
        "mapping": {'Effusion': 'Effusion'} # 공통 질병 매핑
    },
    "CHEXPERT": {
        "train_dir": os.path.join(BASE_PATH, "chexpert_train_320"),
        "val_dir": os.path.join(BASE_PATH, "chexpert_val_320"),
        "train_csv": os.path.join(BASE_PATH, "chexpert_train_320.csv"),
        "val_csv": os.path.join(BASE_PATH, "chexpert_val_320.csv"),
        "labels": ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 
                   'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 
                   'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'],
        "mapping": {'Effusion': 'Pleural Effusion'} # CheXpert는 명칭이 다름
    }
}

BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

# --- 2. 데이터셋 클래스 ---
class ChestXrayDataset(Dataset):
    def __init__(self, df, img_dir, labels, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['Zip_Entry_Name'])
        
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        label = torch.tensor(row[self.labels].values.astype('float32'))
        return image, label

# --- 3. 데이터 준비 함수 ---
def prepare_data(mode):
    cfg = CONFIG[mode]
    train_df = pd.read_csv(cfg['train_csv'])
    val_df = pd.read_csv(cfg['val_csv'])
    
    if mode == "NIH":
        for lbl in cfg['labels']:
            train_df[lbl] = train_df['Finding Labels'].map(lambda x: 1.0 if lbl in str(x) else 0.0)
            val_df[lbl] = val_df['Finding Labels'].map(lambda x: 1.0 if lbl in str(x) else 0.0)
    else:
        # CheXpert: U-Ones 전략 (-1을 1로)
        train_df[cfg['labels']] = train_df[cfg['labels']].fillna(0).replace(-1, 1)
        val_df[cfg['labels']] = val_df[cfg['labels']].fillna(0).replace(-1, 1)
        
    return train_df, val_df, cfg

# --- 4. 모델 생성 ---
def get_model(num_classes):
    model = models.densenet121(weights='IMAGENET1K_V1')
    for param in model.parameters():
        param.requires_grad = False
    
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, num_classes)
    )
    return model.to(DEVICE)

# --- 5. 실험 실행 (학습 및 평가) ---
def run_experiment():
    train_df, val_df, cfg = prepare_data(DATASET_MODE)
    labels = cfg['labels']
    
    train_trans = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    val_trans = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(ChestXrayDataset(train_df, cfg['train_dir'], labels, train_trans), 
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(ChestXrayDataset(val_df, cfg['val_dir'], labels, val_trans), 
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = get_model(len(labels))
    
    # 가중치 계산 (Log-scale pos_weight)
    pos_counts = torch.tensor(train_df[labels].sum().values).float().to(DEVICE)
    pos_weight = torch.log1p((len(train_df) - pos_counts) / (pos_counts + 1e-6))
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)
    scaler = torch.amp.GradScaler('cuda')

    # 공통 질병 매핑 정보 준비
    common_mapping = {d: (cfg['mapping'][d] if d in cfg['mapping'] else d) for d in COMMON_5}
    common_idxs = {d: labels.index(common_mapping[d]) for d in COMMON_5}

    history = {
        'train_loss': [], 'val_loss': [], 'val_auc': [],
        **{f'auc_{d}': [] for d in COMMON_5} # 개별 질병 AUC 저장 공간
    }

    print(f"[{DATASET_MODE}] 학습 시작 (Device: {DEVICE})")
    best_auc = 0
    
    for epoch in range(NUM_EPOCHS):
        # Epoch 2부터 Unfreeze
        if epoch == 2:
            print(">> Unfreezing all layers...")
            for param in model.parameters(): param.requires_grad = True
            for param_group in optimizer.param_groups: param_group['lr'] = LEARNING_RATE * 0.1

        # --- Training ---
        model.train()
        t_loss = 0
        for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch} Train"):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            t_loss += loss.item()

        # --- Validation ---
        model.eval()
        v_loss = 0
        all_lbls, all_probs = [], []
        with torch.no_grad():
            for imgs, lbls in tqdm(val_loader, desc=f"Epoch {epoch} Val"):
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                with torch.amp.autocast('cuda'):
                    outputs = model(imgs)
                    v_loss += criterion(outputs, lbls).item()
                all_lbls.append(lbls.cpu().numpy())
                all_probs.append(torch.sigmoid(outputs).cpu().numpy())

        all_lbls, all_probs = np.vstack(all_lbls), np.vstack(all_probs)
        
        # --- Metrics ---
        # 전체 평균 AUC
        valid_aucs = [roc_auc_score(all_lbls[:, i], all_probs[:, i]) 
                      for i in range(len(labels)) if len(np.unique(all_lbls[:, i])) > 1]
        mean_auc = np.mean(valid_aucs)
        
        # 히스토리 기록
        history['train_loss'].append(t_loss/len(train_loader))
        history['val_loss'].append(v_loss/len(val_loader))
        history['val_auc'].append(mean_auc)
        
        # 공통 질병별 AUC 기록
        for d, idx in common_idxs.items():
            d_auc = roc_auc_score(all_lbls[:, idx], all_probs[:, idx])
            history[f'auc_{d}'].append(d_auc)

        scheduler.step(v_loss/len(val_loader))
        print(f"Epoch {epoch+1} | T-Loss: {t_loss/len(train_loader):.4f} | V-AUC: {mean_auc:.4f}")

        if mean_auc > best_auc:
            best_auc = mean_auc
            torch.save(model.state_dict(), f"{DATASET_MODE.lower()}_best_model.pth")

    # 최종 결과 저장
    with open(f"{DATASET_MODE.lower()}_history.pkl", 'wb') as f:
        pickle.dump(history, f)
    
    return history

# --- 6. 결과 시각화 함수 ---
def plot_results(nih_hist_path, chexpert_hist_path):
    with open(nih_hist_path, 'rb') as f: nih = pickle.load(f)
    with open(chexpert_hist_path, 'rb') as f: chex = pickle.load(f)
    
    epochs = range(1, len(nih['train_loss']) + 1)
    
    plt.figure(figsize=(15, 10))
    
    # 1. Loss 비교
    plt.subplot(2, 2, 1)
    plt.plot(epochs, nih['val_loss'], 'r--', label='NIH Val Loss')
    plt.plot(epochs, chex['val_loss'], 'b-', label='CheXpert Val Loss')
    plt.title('Validation Loss')
    plt.legend()

    # 2. Mean AUC 비교
    plt.subplot(2, 2, 2)
    plt.plot(epochs, nih['val_auc'], 'r--', label='NIH Mean AUC')
    plt.plot(epochs, chex['val_auc'], 'b-', label='CheXpert Mean AUC')
    plt.title('Overall Mean AUC')
    plt.legend()

    # 3. 공통 질병 AUC 비교 (예시: Atelectasis)
    plt.subplot(2, 2, 3)
    plt.plot(epochs, nih['auc_Atelectasis'], 'r--', label='NIH Atelectasis')
    plt.plot(epochs, chex['auc_Atelectasis'], 'b-', label='CheXpert Atelectasis')
    plt.title('Atelectasis AUC Comparison')
    plt.legend()

    # 4. 공통 질병 AUC 비교 (예시: Cardiomegaly)
    plt.subplot(2, 2, 4)
    plt.plot(epochs, nih['auc_Cardiomegaly'], 'r--', label='NIH Cardiomegaly')
    plt.plot(epochs, chex['auc_Cardiomegaly'], 'b-', label='CheXpert Cardiomegaly')
    plt.title('Cardiomegaly AUC Comparison')
    plt.legend()
    
    plt.tight_layout()
    plt.show()


In [ ]:
# 1. NIH 학습
# DATASET_MODE = "NIH"
# nih_history = run_experiment()

# 2. CheXpert 학습 (필요 시 주석 해제하여 순차 진행)
DATASET_MODE = "CHEXPERT"
chex_history = run_experiment()